<p style="text-align: right; margin: 0; line-height: 0;"><img width="100px" height="auto" src="../../.admin/img/eth_logo.png"></p>

## 05.12 Understanding Reasoning Models

<div class="alert alert-light">

  **Name:** (add your name here)
  
  **Matriculation Number:** (add your number here)
  
</div>


### 1. Introduction
So far you have seen how model parameters control how a model speaks and generates. There is another dimension that matters just as much: **how models think**. Many reasoning tasks require the model to plan, connect ideas, and work in steps before it gives a final answer. This process is called **Chain of Thought** (CoT) reasoning. In this notebook we explore how to make smaller local models reason in a visible, useful way and how to test and compare that behavior.

### 2. Prompting methods: imitating CoT
Chain of Thought is a reasoning trace. It is a short sequence of intermediate steps that the model writes before the final answer. You can treat it as a visible scratchpad that keeps the model organized and lets you judge whether the logic makes sense.

As we learned in [05.05_Introduction_to_LLMs.ipynb](05.05_Introduction_to_LLMs.ipynb), Large Language Models learn statistical patterns from huge corpora. At inference time the model does not solve problems analytically. It predicts the next token that best continues the current context. 
When you append the phrase “Think step by step” to a prompt, you steer the model towards a distribution that was trained on many examples where hard problems were explained in steps. 
Thus, the adjusted prompt pulls the output distribution toward those examples and away from one-liners. The model is still doing next-token prediction, but you have changed the target style. It now aims to produce a multi-step answer that ends in a conclusion.

#### 2.1 Simple CoT prompting

<div class="alert alert-block alert-info">

Run the following commands and compare their outputs:

</div>

```sh
ollama run gemma3:1b-it-qat "What is 37 multiplied by 24?"

ollama run gemma3:1b-it-qat "What is 37 multiplied by 24? Think step by step."
```

The second command does more than add words. The prompt leads the model to organize its continuation into a small plan that leads to the result. This difference is not just output; it is a fundamental change in how the model reasons internally.


#### 2.2 Few-shot CoT prompting

We can push the CoT imitation further with **few-shot** CoT. For this, we provide a compact example that contains a rationale and a final answer. We then ask the new question to make the model imitate both format and depth.

<div class="alert alert-block alert-info">

Run the following command and compare it to the previous commands:

</div>

```sh
ollama run gemma3:1b-it-qat "Q: If a pen costs 2 and a notebook costs 5, how much for 3 pens and 2 notebooks?
A: First compute 3×2 = 6 for pens. Then 2×5 = 10 for notebooks. Add 6 + 10 = 16.
Final: 16

With this example solve the next question and think step by step.
Q: A box has 8 red and 5 blue balls. If I add 7 red, how many balls total?"
```


#### 2.3 Structured few-shot CoT prompting

We can also **structure** the output so that it stays machine-checkable and avoids rambling. 

<div class="alert alert-block alert-info">

Run the following command and compare it to the previous commands:

</div>

```sh
ollama run gemma3:1b-it-qat "Q: If a pen costs 2 and a notebook costs 5, how much for 3 pens and 2 notebooks?
A: First compute 3×2 = 6 for pens. Then 2×5 = 10 for notebooks. Add 6 + 10 = 16.
Final: 16

With this example solve the next question and think step by step.
Follow this EXACT output format:
<THINK>...your brief reasoning, 2–4 sentences...</THINK>
Answer: <VALUE>

Rules:
- Use <THINK> ... </THINK> with a proper closing tag.
- The final line must start with: Answer: 
- Replace <VALUE> with a single integer or decimal.
- Do not print anything after the Answer line. No extra lines, no quotes.

Q: A box has 8 red and 5 blue balls. If I add 7 red, how many balls total?"
```

All of these are **prompting** methods. We did not change the weights, the model, intrinsics, ... we simply arranged the context so that next-token prediction favors concise, stepwise explanations that end in a clean final line. 



### 3. CoT: Reasoning models
A reasoning model is trained not just to produce answers but to explain them. Instead of learning the mapping "Question -> Answer", it specifically learns "Question -> Reasoning -> Answer". That extra supervision changes how it organizes the next-token predictions. When such a model sees a question, it doesn't jump straight to a solution; it first drafts a small internal plan.

Training this behavior can be done in several ways.
The simplest is **supervised fine-tuning with rationales**. We show the model thousands of examples that contain both intermediate reasoning and the correct final result.
The model internalizes that good answers include a thought process.
A more refined version is called **process supervision**. Instead of rewarding only the final answer, every intermediate reasoning step is evaluated. The model learns how to think, instead of just what to output.
Another approach is **rejection sampling**. During training, the model generates multiple reasoning traces for each question. These are then evaluated and only those leading to the correct outcome are kept for training. The model absorbs the structure of reasoning paths that actually succeed.

Newer techniques like **Direct Preference Optimization (DPO)** or **Reinforcement Learning from Human Feedback (RLHF)** go beyond standard fine-tuning. They do not just show the model examples, they actively teach it what good reasoning looks like by comparing and ranking possible answers.

With **RLHF**, a separate reward model is trained from human judgments on the completion quality. The model then learns to maximize the reward from that feedback, gradually preferring explanations that are accurate, structured, and easy to follow. It is like a teacher marking which solutions were better written, not just which were numerically right.

**DPO** achieves a similar effect without heavy reinforcement-learning. We start with a base model that can produce multiple possible answers to a question. Then, we collect pairs of completions for the same prompt, one that humans preferred and one that lost the comparison. DPO then gently reshapes the distribution so the model leans towards patterns found in good reasoning traces. This keeps training stable and efficient while still capturing preference signals such as *"short and clear chains are better than long and confusing chains*".

Generally, RLHF scales extremely well because once a **strong reward model** has been trained, it can automatically score millions of model outputs without needing new human labels. DPO on the other hand dominates among smaller open-source models because it is simpler and cheaper.

Together, these methods shape a network that treats reasoning as a natural part of communication rather than just a stylistic choice. They make the model internally align its token probabilities with our sense of what good reasoning sounds like.
Under the hood, reasoning models mostly preserve the same architecture as their non-reasoning counterparts. Because of this, even small models, such as *deepseek-r1:1.5b* or *qwen3:1.7b*, can display surprisingly mature reasoning if their fine-tuning emphasized clean examples and compact steps.

**Further readings**:

[Wei et al., 2022 "Chain-of-Thought Prompting Elicits Reasoning in Large Language Models"](https://arxiv.org/pdf/2201.11903)

[Rafailov et al., 2023 "Direct Preference Optimization: Your Language Model is Secretly a Reward Model"](https://arxiv.org/pdf/2305.18290)

[Christiano et al., 2017 "Deep Reinforcement Learning from Human Preferences"](https://arxiv.org/pdf/1706.03741)

[DeepSeek-AI, 2025 "DeepSeek-R1: Incentivizing Reasoning Capability in LLMs via Reinforcement Learning"](https://arxiv.org/pdf/2501.12948)

[Qwen Team, 2025 "Qwen3 Technical Report"](https://arxiv.org/pdf/2505.09388)



### 4. Model comparison
We now compare reasoning behavior across models of different sizes and training styles. Some rely entirely on their size and training diversity, others have been fine-tuned to reason explicitly.

#### 4.1 Same task, different scale

<div class="alert alert-block alert-info">

Run the following commands and compare:

</div>


```sh
ollama run gemma3:270m "A factory produces 45 parts per hour. After working for 4 hours, the production rate increases by 20%. How many total parts are produced in 8 hours?"

ollama run gemma3:1b-it-qat "A factory produces 45 parts per hour. After working for 4 hours, the production rate increases by 20%. How many total parts are produced in 8 hours?"
```

Both models share architecture and tokenizer. Their difference comes purely from capacity and training diversity. The 270M model will likely fragment or guess, while the 1B can sustain a short reasoning chain, even if the arithmetic itself is off.

#### 4.2 Structured reasoning prompt

Now, apply a few-shot structured prompt to both models:

<div class="alert alert-block alert-info">

Run the following command for the two models (`gemma3:270m`, `gemma3:1b-it-qat`) and compare:

</div>

```sh
ollama run <MODEL> "Q: A machine makes 30 screws per minute. After running for 10 minutes, its speed increases by 50%. How many screws are made in 20 minutes?
A: For the first 10 minutes, the machine makes 30 × 10 = 300 screws. 
After that, speed increases by 50%, so the new rate is 30 × 1.5 = 45 screws per minute. 
In the next 10 minutes, it makes 45 × 10 = 450 screws. 
Add them: 300 + 450 = 750.
Final: 750

With this example solve the next question and think step by step.
Follow this EXACT output format:
<THINK>...your brief reasoning, 2–4 sentences...</THINK>
Answer: <VALUE>

Rules:
- Use <THINK> ... </THINK> with a proper closing tag.
- The final line must start with: Answer: 
- Replace <VALUE> with a single integer or decimal.
- Do not print anything after the Answer line. No extra lines, no quotes.
Q: A factory produces 45 parts per hour. After working for 4 hours, the production rate increases by 20%. How many total parts are produced in 8 hours? Think step by step."
```

With the instructions in place, you should now see more structured reasoning, though the smaller model may still lose consistency.

#### 4.2.1 Behavioral contracts
Many models accept a **system message** to give behavior guidelines that load before the user prompt. In case a model *(e.g. Gemma)* does not support it, Ollama effectively prepends it to the prompt. Either way, it is the proper place for behavior contracts so your prompt stays clean.


```sh
ollama-gen --model qwen3:1.7b --think false --system "Solve this problem by thinking out loud. Work step-by-step. Format your output with <THINK> tags and <ANSWER> tags." --prompt "A farmer has 7 cows and 3 chickens. How many footprints occur per step, including his own?"
```

We can now go a step further and bring this into a familiar output format. By introducing the `--format` flag and setting its value to `"json"`, we can instruct Ollama to output a valid JSON object. This keeps JSON-keys stable and makes the output more manageable for automated processes.

```sh
ollama-gen --model qwen3:1.7b --think false --system "Solve this problem by thinking out loud. Work step-by-step. Format your output in a JSON object with the keys 'think' and 'answer'" --prompt "A farmer has 7 cows and 3 chickens. How many footprints occur per step, including his own?" --format "json"
```

#### 4.3 Built-in thinking vs prompted thinking


Modern reasoning models like *Qwen3* allow toggling an internal *thinking mode*. This can produce deeper, but longer reasoning traces.

<div class="alert alert-block alert-info">

Run the following commands and compare:

</div>

```sh
ollama run qwen3:1.7b "A factory produces 45 parts per hour. After working for 4 hours, the production rate increases by 20%. How many total parts are produced in 8 hours?" --think=false

ollama run qwen3:1.7b "A factory produces 45 parts per hour. After working for 4 hours, the production rate increases by 20%. How many total parts are produced in 8 hours?" --think
```

> *Note:* Feel free to terminate the generation at any time. We do not collect metrics. Just look at the reasoning and projected output solution.

The thinking variant often generates long self-talking chains. While the reasoning is mostly concise and numerically correct, it does start to drift and go in circles.
As investigated in [HandsOn Inference Control, Section 2.1](05.11_Understanding_Inference_Control.ipynb), *Qwen is optimized for open-ended, multilingual, and code-heavy continuations. This gives it greater variation but also makes it more prone to long, free-flowing outputs if unconstrained.*
The non-thinking version is faster and cleaner, but its explanations remain formulaic and cryptic at times.

#### 4.4 Comparing model families
Finally, we test reasoning across architectures:

<div class="alert alert-block alert-info">

Choose any complex arithmetic or logic problem (example below) and compare across models:

</div>

```sh
ollama run qwen3:1.7b "<PROMPT>"

ollama run qwen3:4b "<PROMPT>"

ollama run deepseek-r1:1.5b "<PROMPT>"
```

**Example prompt:** 
"A cargo train covers 180 kilometers in 3 hours while pulling 30 wagons.

After a scheduled stop, 10 additional wagons are attached,
and the train's average speed decreases by 15 % due to the added load.

If the train must cover a total distance of 480 kilometers that day,
how many additional hours are required after the stop to complete the trip?"

**Expected answer**: around 5.88 h


In most cases, you will struggle with seemingly endless rambling in the thinking phase. This is Qwen's internal reasoning trace made visible. The model is practically writing out different paths it explores before deciding on an outcome. This happens because its thinking mode prioritizes completeness and self-reflection. To solve this, you can either turn off the thinking altogether with `--think=false` or you can terminate (`CTRL+C`) the text generation at any point. 

### 5. Modern techniques and outlook
Chain-of-Thought reasoning is only the foundation. Once a model can make its thinking explicit, more advanced techniques can build on it to create deeper and more reliable reasoning.
Together, these methods illustrate how reasoning evolves from a single visible chain into an organized process. The benefits from these patterns can be used at every scale. Even small local models can be used for running multiple reasoning paths, reflecting on their outputs, or grounding their steps in retrieved evidence.

#### 5.1 Self-Consistency
**Self-Consistency** takes advantage of the natural variability in a model’s outputs. By sampling **multiple reasoning traces** for the same problem and comparing their conclusions, we can vote on the most common final answer. This simple **ensemble approach** cancels random drift and often results in higher accuracy without any retraining.

Further reading: [Wang et al., 2022 "Self-Consistency Improves Chain of Thought Reasoning in Language Models"](https://arxiv.org/pdf/2203.11171)

#### 5.2 Tree-of-Thought (ToT)
**ToT** extends this idea by exploring several reasoning **branches in parallel**. Instead of committing to a single chain, the model treats reasoning as a search tree. Each branch represents a possible line of thought, and weaker paths are pruned as the model progresses. This allows the model to **recover from early mistakes** and combine partial insights from different routes before deciding on a final answer.

Further reading: [Yao et al., 2023 "Tree of Thoughts: Deliberate Problem Solving with Large Language Models"](https://arxiv.org/pdf/2305.10601)

#### 5.3 Reflexion
**Reflexion** adds self-evaluation. After completing a reasoning trace, the model **reviews its own work, critiques potential errors, and attempts an improved solution**. This feedback loop transforms generation into an iterative process during which the model not only answers but also learns to refine its reasoning within a single session.

Further reading: [Shinn et al., 2023 "Reflexion: Language Agents with Verbal Reinforcement Learning"](https://arxiv.org/pdf/2303.11366)

#### 5.4 Retrieval-Augmented Generation with Chain-of-Thought
**RAG + CoT** merges reasoning with memory. Here the model first retrieves factual information from local or external sources, then applies step-by-step reasoning to those facts. This separation of **recall and reasoning** improves factual accuracy and transparency, especially for tasks that mix computation with world knowledge.

Further reading: [Li et al., 2025 "CoT-RAG: Integrating Chain of Thought and Retrieval-Augmented Generation to Enhance Reasoning in Large Language Models"](https://arxiv.org/pdf/2504.13534)



----
### Questions

<div class="alert alert-block alert-info">Fill in your responses and save this notebook. See the <a href="../../03_Assignments/Submission-Guidelines.md">submission guidelines</a> for how to submit your work.</div>

**In your own words, what is the main idea behind CoT reasoning and how doe models differ from normal/regular LLMs?** Answer:

(add your answer here)

**Which step of the CoT prompting techniques yielded the best results? Were the structuring rules always respected?** Answer:

(add your answer here)

**Why might structured prompting with tags like *<THINK>* and *Answer* be useful?** Answer:

(add your answer here)

**What is the difference between supervised fine-tuning with rationales and process supervision?** Answer:

(add your answer here)

**You saw few-shot. What does zero-shot mean? When could this be beneficial?** Answer:

(add your answer here)

**Why do you think the DPO paper is called "[...] Your Language Model is Secretly a Reward Model"? Aren't reward models a thing in RLHF?** Answer:

(add your answer here)

**How did the Gemma3 270M model compare against the 1B-it-qat model? How was it after introducing the CoT prompting instructions?** Answer:

(add your answer here)

**How did the Qwen3 1.7B model perform? What happened when you disabled thinking?** Answer:

(add your answer here)

**How did the Qwen3 and DeepSeek-r1 model families compare?** Answer:

(add your answer here)

**Does DeepSeek-r1 1.5B also suffer from rambling? Check its architecture with `ollama show` and try to reason why that may be.** Answer:

(add your answer here)

**Briefly explain how feasible and useful each of the modern techniques are for small local models.** Answer:

(add your answer here)

**Which model has proven to you personally that it edges out the others?** Answer:

(add your answer here)

**OPTIONAL: Did you read through and understand all the tasks in this Notebook? Are there any shortcomings, missing parts, or sections that should be improved?**

 (please help the team by telling us what part was not understood, and we will work on it)